## Train the Poisson regression model

### 1. Hyperparameter tuning

Each Optuna trial picks a value for every hyperparameter, trains the model on the 3 validation folds,
and returns the mean Poisson deviance (lower is better). <br> Optuna uses the results to pick better values in the next trials.

The test set is not used here.

In [1]:
import sys
from pathlib import Path

import optuna

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utilities.poisson_regression import *
from utilities.training import *
from utilities.utilities import *
from utilities.values import *

data_dir = '../data/'
model_dir = '../models/'

# read the train-validation set (Jan 2020 - Feb 2021)
train_validation_df = read_parquet_file(data_dir + 'processed/train_validation.parquet')

In [2]:
def objective(trial):
    # 1. let optuna pick the hyperparams for this trial
    hyperparams = {
        'alpha': trial.suggest_float('alpha', 1e-6, 1.0, log = True),   # search on a log scale
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cholesky']),
    }

    # 2. train a new model with these hyperparams on each validation fold, and score it
    scores_df = cross_validate(PoissonRegressionModel, hyperparams, train_validation_df)
    mean_scores = scores_df[['MAE', 'RMSE', 'poisson_deviance']].mean()

    # Optuna tunes on the Poisson deviance only
    # MAE and RMSE are stored with the trial so they also appear in the results table
    trial.set_user_attr('MAE', mean_scores['MAE'])
    trial.set_user_attr('RMSE', mean_scores['RMSE'])

    # show the scores of this trial
    print('Poisson deviance per fold:', scores_df['poisson_deviance'].round(4).tolist())
    print(f'Trial {trial.number}: mean Poisson deviance = {mean_scores["poisson_deviance"]:.4f}, '
          f'mean MAE = {mean_scores["MAE"]:.4f}, mean RMSE = {mean_scores["RMSE"]:.4f}\n')
    return mean_scores['poisson_deviance']


# hide Optuna's own log line for every trial (the progress bar below shows the progress instead)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# the seed makes the search repeatable
study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler(seed = OPTUNA_SEED))

In [3]:
n_trials = 30

# the progress bar shows how many trials are done, the time remaining and the best score so far
study.optimize(objective, n_trials = n_trials, show_progress_bar = True)

print('Best hyperparameters:', study.best_params)
print('Best mean Poisson deviance:', round(study.best_value, 4))
print('Mean MAE of the best trial:', round(study.best_trial.user_attrs['MAE'], 4))
print('Mean RMSE of the best trial:', round(study.best_trial.user_attrs['RMSE'], 4))

  0%|          | 0/30 [00:00<?, ?it/s]

Fold 1: training PoissonRegressionModel on 298605 rows
Fold 2: training PoissonRegressionModel on 373269 rows
Fold 3: training PoissonRegressionModel on 447933 rows
Poisson deviance per fold: [1.1997, 1.036, 0.6366]
Trial 0: mean Poisson deviance = 0.9574, mean MAE = 0.5938, mean RMSE = 0.9748

Fold 1: training PoissonRegressionModel on 298605 rows
Fold 2: training PoissonRegressionModel on 373269 rows
Fold 3: training PoissonRegressionModel on 447933 rows
Poisson deviance per fold: [1.2418, 1.0381, 0.6517]
Trial 1: mean Poisson deviance = 0.9772, mean MAE = 0.6033, mean RMSE = 0.9522

Fold 1: training PoissonRegressionModel on 298605 rows
Fold 2: training PoissonRegressionModel on 373269 rows
Fold 3: training PoissonRegressionModel on 447933 rows
Poisson deviance per fold: [1.1987, 1.0355, 0.6357]
Trial 2: mean Poisson deviance = 0.9566, mean MAE = 0.5929, mean RMSE = 0.9786

Fold 1: training PoissonRegressionModel on 298605 rows
Fold 2: training PoissonRegressionModel on 373269 rows


In [4]:
# all trials, best first
trials_df = study.trials_dataframe()
trials_df.sort_values('value').head(10)

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_solver,user_attrs_MAE,user_attrs_RMSE,state
14,14,0.956542,2026-10-24 02:56:25.973124,2026-10-24 02:56:57.383939,0 days 00:00:31.410815,0.000044,lbfgs,0.593089,0.977581,COMPLETE
15,15,0.956561,2026-10-24 02:56:57.385097,2026-10-24 02:57:27.830958,0 days 00:00:30.445861,0.000062,lbfgs,0.593173,0.976912,COMPLETE
21,21,0.956596,2026-10-24 02:58:51.174469,2026-10-24 02:59:27.045303,0 days 00:00:35.870834,0.000006,lbfgs,0.592862,0.978721,COMPLETE
23,23,0.956605,2026-10-24 03:00:14.216157,2026-10-24 03:00:58.034511,0 days 00:00:43.818354,0.000040,lbfgs,0.593126,0.977708,COMPLETE
2,2,0.956633,2026-10-24 02:53:47.814466,2026-10-24 02:54:25.669012,0 days 00:00:37.854546,0.000002,lbfgs,0.592942,0.978627,COMPLETE
16,16,0.956641,2026-10-24 02:57:27.831679,2026-10-24 02:57:59.095412,0 days 00:00:31.263733,0.000070,lbfgs,0.593185,0.976790,COMPLETE
26,26,0.956760,2026-10-24 03:01:27.595583,2026-10-24 03:02:04.753427,0 days 00:00:37.157844,0.000005,lbfgs,0.592987,0.978867,COMPLETE
12,12,0.956787,2026-10-24 02:56:14.396669,2026-10-24 02:56:20.009973,0 days 00:00:05.613304,0.000033,newton-cholesky,0.593206,0.978036,COMPLETE
11,11,0.956839,2026-10-24 02:56:08.464385,2026-10-24 02:56:14.395534,0 days 00:00:05.931149,0.000020,newton-cholesky,0.593162,0.978465,COMPLETE
5,5,0.956896,2026-10-24 02:54:34.550593,2026-10-24 02:54:40.526753,0 days 00:00:05.976160,0.000013,newton-cholesky,0.593139,0.978720,COMPLETE


### 2. Validation predictions (for the ensemble)

- Get the best set of hyperparameters from optuna. 
- For each of the cross-validation sets, train the model on the train set, and store the validation set predictions.
- These validation set predictions will be used to train the ensemble model later.

In [5]:
# get the predictions of each validation fold
fold_results = get_out_of_fold_predictions(PoissonRegressionModel, study.best_params, train_validation_df)

# save the predictions in each file separately
for fold_number, fold_df in enumerate(fold_results, start = 1):
    save_to_parquet_file(fold_df, data_dir + f'predictions/validation_poisson_regression_fold{fold_number}.parquet')
print('Validation predictions saved.')

Fold 1: training PoissonRegressionModel on 298605 rows
Fold 2: training PoissonRegressionModel on 373269 rows
Fold 3: training PoissonRegressionModel on 447933 rows
Validation predictions saved.


### 3. Final model and test set prediction

In [6]:
test_df = read_parquet_file(data_dir + 'processed/test.parquet')

In [7]:
# train a new model on the entire train set (Jan 2020 - Feb 2021) using the best hyperparams from Optuna
model = PoissonRegressionModel(**study.best_params)
model.fit(train_validation_df)

# do predictions on the test set (Mar -April 2021)
test_df[PREDICTION] = model.predict(test_df)

# save the model and results
model.save_model(model_dir + 'poisson_regression_model.pkl')
save_to_parquet_file(test_df, data_dir + 'predictions/test_poisson_regression.parquet')
print('Model and test predictions saved.')

Model and test predictions saved.


In [11]:
test_df.head()

,date,num_rentals,start station name,hour_of_day,day_of_week,num_rentals_t-2,num_rentals_t-3,sine_week_of_year,cosine_week_of_year,temperature_2m,snow_depth,cloud_cover,cloud_cover_mid,relative_humidity_2m,precipitation,wind_speed_10m,is_public_holiday,prediction
0,2021-03-01,0,5 Corners Library,0,0,0.0,0.0,0.812487,0.582979,5.55,0.1,100.0,99.0,96.586780,0.5,10.787993,0,0.024940
1,2021-03-01,0,5 Corners Library,1,0,0.0,0.0,0.812487,0.582979,5.40,0.1,100.0,100.0,97.596886,0.7,12.181624,0,0.011928
2,2021-03-01,0,5 Corners Library,2,0,0.0,0.0,0.812487,0.582979,5.45,0.1,100.0,100.0,97.597850,0.4,9.164103,0,0.007014
3,2021-03-01,0,5 Corners Library,3,0,0.0,0.0,0.812487,0.582979,5.45,0.1,100.0,100.0,97.597850,0.5,6.952755,0,0.004683
4,2021-03-01,0,5 Corners Library,4,0,0.0,0.0,0.812487,0.582979,5.05,0.1,100.0,100.0,98.273410,1.0,8.121970,0,0.005985
